## Inspect the CUDA runtime

Run the notebook on a GPU runtime with `nvcc` available. These commands show the assigned GPU and the installed CUDA compiler.


In [ ]:
!nvidia-smi
!nvcc --version
!mkdir -p code

## Write the vector-addition program

The kernel and host program are kept together in one CUDA source file. Each thread computes one element; the bounds check handles a final partially filled block. The host arrays and device arrays use the same pointer-based layout.


In [ ]:
%%writefile code/chapter01_vector_add_cuda.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define CUDA_CHECK(call) do {                                      \
    cudaError_t error = (call);                                    \
    if (error != cudaSuccess) {                                    \
        fprintf(stderr, "%s:%d: %s failed: %s\n",                \
                __FILE__, __LINE__, #call,                         \
                cudaGetErrorString(error));                        \
        exit(EXIT_FAILURE);                                        \
    }                                                               \
} while (0)

__global__ void vector_add(const float *a, const float *b,
                           float *c, int n)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

int main(int argc, char **argv)
{
    int n = 19;
    if (argc > 2) {
        fprintf(stderr, "Usage: %s [positive_length]\n", argv[0]);
        return EXIT_FAILURE;
    }
    if (argc == 2) {
        n = atoi(argv[1]);
    }
    if (n < 1) {
        fprintf(stderr, "The vector length must be positive.\n");
        return EXIT_FAILURE;
    }

    const size_t bytes = (size_t)n * sizeof(float);
    float *a = (float *)malloc(bytes);
    float *b = (float *)malloc(bytes);
    float *c = (float *)malloc(bytes);
    if (a == NULL || b == NULL || c == NULL) {
        fprintf(stderr, "Could not allocate host arrays.\n");
        free(a);
        free(b);
        free(c);
        return EXIT_FAILURE;
    }

    for (int i = 0; i < n; ++i) {
        a[i] = 0.5f * i;
        b[i] = 1.0f + 0.25f * (i % 7);
        c[i] = 0.0f;  // Sentinel: detects an output element not written by the kernel.
    }

    float *d_a = NULL;
    float *d_b = NULL;
    float *d_c = NULL;
    CUDA_CHECK(cudaMalloc(&d_a, bytes));
    CUDA_CHECK(cudaMalloc(&d_b, bytes));
    CUDA_CHECK(cudaMalloc(&d_c, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    int failures = 0;
    for (int i = 0; i < n; ++i) {
        if (c[i] != a[i] + b[i]) {
            ++failures;
        }
    }

    printf("N = %d, threads per block = %d, blocks = %d\n",
           n, threads, blocks);
    printf("Threads without a valid element: %d\n\n", blocks * threads - n);
    printf("     i          a[i]          b[i]       GPU c[i]       CPU a+b\n");
    for (int i = 0; i < n; ++i) {
        if (n <= 20 || i < 8 || i >= n - 4) {
            printf("%6d %13.4f %13.4f %14.4f %13.4f\n",
                   i, a[i], b[i], c[i], a[i] + b[i]);
        } else if (i == 8) {
            printf("   ... (remaining elements are also checked) ...\n");
        }
    }
    printf("\nValidation: %s (%d failures among %d elements)\n",
           failures == 0 ? "PASS" : "FAIL", failures, n);

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    free(a);
    free(b);
    free(c);
    return failures == 0 ? EXIT_SUCCESS : EXIT_FAILURE;
}


## Compile

This is the same command used in the chapter. `-O2` enables optimization, `-std=c++14` selects the C++ language version, and `-arch=native` targets the GPU assigned to the runtime.


In [ ]:
!nvcc -O2 -std=c++14 -arch=native code/chapter01_vector_add_cuda.cu -o vector_add_cuda

## Run the program

The default length is 19. Every output is checked against the CPU sum; these values are exactly representable in single precision.


In [ ]:
!./vector_add_cuda 19

## Check a length that crosses a block boundary

With 259 elements and 256 threads per block, the grid has two blocks. The kernel launches 512 threads, so 253 have no valid element. The program still checks all 259 outputs.


In [ ]:
!./vector_add_cuda 259